# 02 — Tiền xử lý & Xử lý Imbalanced Data
**Phụ trách:** Hoàng  
**Mục tiêu:** Feature Engineering → SMOTE → Lưu data/processed/

> Chạy notebook này trước `03_modeling.ipynb`

In [ ]:
import sys; sys.path.append('..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from src.data_loader import load_data
from src.preprocess import (engineer_features, filter_fraud_types,
    split_features_target, split_train_test, apply_smote)
sns.set_theme(style='whitegrid')
print('Setup OK')

## 1. Load dữ liệu thô

In [ ]:
# Dùng nrows khi test nhanh, xoá khi train chính thức
df = load_data(nrows=500_000)
df.head()

## 2. Feature Engineering

Tạo `errorBalanceOrig` và `errorBalanceDest` — sai lệch logic số dư
(insight quan trọng nhất từ EDA: fraud thường drain sạch tài khoản)

In [ ]:
df_feat = engineer_features(df)
print('Columns:', df_feat.columns.tolist())
df_feat.head()

In [ ]:
# Kiểm tra: fraud có errorBalance cao hơn legit không?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['errorBalanceOrig', 'errorBalanceDest']):
    ax.hist(np.log1p(np.abs(df_feat[df_feat.isFraud==0][col].sample(5000))),
            bins=50, alpha=0.6, label='Legit', color='steelblue')
    ax.hist(np.log1p(np.abs(df_feat[df_feat.isFraud==1][col])),
            bins=50, alpha=0.6, label='Fraud', color='crimson')
    ax.set_title(f'log(|{col}|)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/error_balance_dist.png', dpi=150)
plt.show()

## 3. Lọc transaction type

Theo EDA: gian lận **chỉ** xảy ra với `TRANSFER` và `CASH_OUT`

In [ ]:
df_f = filter_fraud_types(df_feat)
print(f'Fraud rate sau lọc: {df_f.isFraud.mean()*100:.3f}%')

## 4. Train-Test Split (Stratified)

In [ ]:
X, y = split_features_target(df_f)
X_train, X_test, y_train, y_test = split_train_test(X, y)
print('Features:', X.columns.tolist())

## 5. SMOTE

> **Quan trọng:** Chỉ áp dụng SMOTE trên **tập train**, KHÔNG trên test!
> Áp dụng trên test = data leakage → kết quả ảo.

In [ ]:
X_res, y_res = apply_smote(X_train, y_train)

# Visualize before vs after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (y_d, title) in zip(axes,[(y_train,'Trước SMOTE'),(y_res,'Sau SMOTE')]):
    c = pd.Series(y_d).value_counts()
    ax.bar(['Legit','Fraud'], c.values, color=['steelblue','crimson'], alpha=0.8)
    ax.set_title(title)
    for i,v in enumerate(c.values): ax.text(i, v*1.01, f'{v:,}', ha='center')
plt.tight_layout()
plt.savefig('../reports/figures/smote_comparison.png', dpi=150)
plt.show()

## 6. Lưu processed data

In [ ]:
import os; os.makedirs('../data/processed', exist_ok=True)
X_res.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_res.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
print('Saved all processed data!')